In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/ieee-fraud-detection/sample_submission.csv
/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv
/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv


In [2]:
pip install dagshub mlflow scikit-learn pandas matplotlib seaborn skops --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.1/273.1 kB 2.1 MB/s eta 0:00:00a 0:00:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 5.1 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 8.6 MB/s eta 0:00:0000:0100:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 9.6 MB/s eta 0:00:00ta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.4/208.4 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━

In [3]:
import numpy as np
import pandas as pd
import mlflow
import mlflow.sklearn
import dagshub
from sklearn.pipeline import Pipeline
from sklearn.ensemble import AdaBoostClassifier
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

dagshub.init(repo_owner='Nestor-Dzadzamia', repo_name='IEEE-CIS-Fraud-Detection', mlflow=True)

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=59424a18-952f-40c5-933d-57abe5d589a6&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=3a010281feff91b55fd9d150ecbf7f052d3cc41b0eedfc25a4dc90b14cdf9fe0




Accessing as Nestor-Dzadzamia

Initialized MLflow to track repo "Nestor-Dzadzamia/IEEE-CIS-Fraud-Detection"

Repository Nestor-Dzadzamia/IEEE-CIS-Fraud-Detection initialized!

# Merge Data

In [4]:
train_transaction = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv')
train_identity = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv')

print(train_transaction.shape)
print(train_identity.shape)

(590540, 394)
(144233, 41)


# Train/Test split

In [5]:
df = train_transaction.merge(train_identity, on='TransactionID', how='left')
print(f"Merged shape: {df.shape}")

y = df['isFraud']
X = df.drop(columns=['isFraud', 'TransactionID'])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"Fraud ratio: {y_train.mean():.4f}")

Merged shape: (590540, 434)
X_train shape: (472432, 432)
X_test shape: (118108, 432)
Fraud ratio: 0.0350


# Cleaning, Feature Engineering & Feature Selection

In [6]:
mlflow.set_experiment("AdaBoost_Training")

with mlflow.start_run(run_name="ADA_Cleaning"):
    cols_dropped = X.columns[X.isnull().mean() > 0.5].tolist()
    mlflow.log_param("null_thresh", 0.5)
    mlflow.log_param("num_imputer_strategy", "median")
    mlflow.log_param("cat_imputer_strategy", "most_frequent")
    mlflow.log_metric("cols_dropped", len(cols_dropped))
    mlflow.log_metric("cols_remaining", X.shape[1] - len(cols_dropped))

with mlflow.start_run(run_name="ADA_Feature_Engineering"):
    mlflow.log_param("new_features", "log_TransactionAmt, Transaction_hour, Transaction_day")
    mlflow.log_metric("original_cols", X.shape[1])
    mlflow.log_metric("new_cols_added", 3)

with mlflow.start_run(run_name="ADA_Feature_Selection"):
    mlflow.log_param("corr_thresh", 0.9)
    mlflow.log_param("methods", "correlation_filter")

print("ADA preprocessing runs logged!")

🏃 View run ADA_Cleaning at: https://dagshub.com/Nestor-Dzadzamia/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/3/runs/026108324de44059ba29d5e96fdb9d80
🧪 View experiment at: https://dagshub.com/Nestor-Dzadzamia/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/3
🏃 View run ADA_Feature_Engineering at: https://dagshub.com/Nestor-Dzadzamia/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/3/runs/37d1beb59af944339dd5695dd7b6c0e1
🧪 View experiment at: https://dagshub.com/Nestor-Dzadzamia/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/3
🏃 View run ADA_Feature_Selection at: https://dagshub.com/Nestor-Dzadzamia/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/3/runs/4d024fc34efb4e73ac78a9ebff994b9c
🧪 View experiment at: https://dagshub.com/Nestor-Dzadzamia/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/3
ADA preprocessing runs logged!


# Creating the preprocessor class

In [7]:
class FraudPreprocessor(BaseEstimator, TransformerMixin):
    def __init__(self, null_thresh=0.5, corr_thresh=0.9):
        self.null_thresh = null_thresh
        self.corr_thresh = corr_thresh

    def _feature_engineering(self, X):
        X = X.copy()
        X['log_TransactionAmt'] = np.log1p(X['TransactionAmt'])
        X['Transaction_hour'] = (X['TransactionDT'] // 3600) % 24
        X['Transaction_day'] = (X['TransactionDT'] // (3600 * 24)) % 7
        return X

    def fit(self, X, y=None):
        X = self._feature_engineering(X)
        
        self.cols_to_drop_ = X.columns[X.isnull().mean() > self.null_thresh].tolist()
        X = X.drop(columns=self.cols_to_drop_)

        corr_matrix = X.select_dtypes(include=np.number).corr().abs()
        upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
        self.corr_cols_to_drop_ = [col for col in upper.columns if any(upper[col] > self.corr_thresh)]
        X = X.drop(columns=self.corr_cols_to_drop_)
        
        self.cat_cols_ = [col for col in X.columns if X[col].dtype == 'object']
        self.num_cols_ = [col for col in X.columns if X[col].dtype != 'object']
        
        self.num_imputer_ = SimpleImputer(strategy='median')
        self.num_imputer_.fit(X[self.num_cols_])
        self.cat_imputer_ = SimpleImputer(strategy='most_frequent')
        self.cat_imputer_.fit(X[self.cat_cols_])
        
        self.encoder_ = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
        X_cat_imputed = self.cat_imputer_.transform(X[self.cat_cols_])
        self.encoder_.fit(X_cat_imputed)
        self.scaler_ = StandardScaler()
        X_num_imputed = self.num_imputer_.transform(X[self.num_cols_])
        
        self.scaler_.fit(X_num_imputed)
        return self

    def transform(self, X):
        X = self._feature_engineering(X)

        # drop correlated and high null columns
        X = X.drop(columns=self.cols_to_drop_)
        X = X.drop(columns=self.corr_cols_to_drop_)

        # filling categorial and numerical columns
        X_num = self.num_imputer_.transform(X[self.num_cols_])
        X_cat = self.cat_imputer_.transform(X[self.cat_cols_])
        
        X_cat_encoded = self.encoder_.transform(X_cat)
        X_num_scaled = self.scaler_.transform(X_num)
        
        X_final = np.hstack([X_num_scaled, X_cat_encoded])
        
        return X_final

# Pipeline and experiments with mlflow logging

In [ ]:
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier

mlflow.set_experiment("AdaBoost_Training")

experiments = [
    {"null_thresh": 0.5, "corr_thresh": 0.9, "n_estimators": 50,  "learning_rate": 1.0, "max_depth": 1},
    {"null_thresh": 0.5, "corr_thresh": 0.9, "n_estimators": 100, "learning_rate": 1.0, "max_depth": 1},
    {"null_thresh": 0.5, "corr_thresh": 0.9, "n_estimators": 100, "learning_rate": 0.5, "max_depth": 2},
    {"null_thresh": 0.7, "corr_thresh": 0.9, "n_estimators": 200, "learning_rate": 0.5, "max_depth": 2},
    {"null_thresh": 0.7, "corr_thresh": 0.9, "n_estimators": 200, "learning_rate": 0.1, "max_depth": 2},
    {"null_thresh": 0.7, "corr_thresh": 0.9, "n_estimators": 200, "learning_rate": 1.0, "max_depth": 3},
]

for exp in experiments:
    pipe = Pipeline(steps=[
        ('preprocessor', FraudPreprocessor(null_thresh=exp["null_thresh"], corr_thresh=exp["corr_thresh"])),
        ('model', AdaBoostClassifier(
            estimator=DecisionTreeClassifier(max_depth=exp["max_depth"]),
            n_estimators=exp["n_estimators"],
            learning_rate=exp["learning_rate"],
            random_state=42
        ))
    ])
    
    pipe.fit(X_train, y_train)
    
    with mlflow.start_run(run_name=f"ADA_n={exp['n_estimators']}_lr={exp['learning_rate']}_depth={exp['max_depth']}_null={exp['null_thresh']}"):
        mlflow.log_params(exp)
        
        y_pred = pipe.predict(X_test)
        y_proba = pipe.predict_proba(X_test)[:, 1]
        y_train_proba = pipe.predict_proba(X_train)[:, 1]
        
        train_roc_auc = roc_auc_score(y_train, y_train_proba)
        test_roc_auc = roc_auc_score(y_test, y_proba)
        mlflow.log_metric("train_roc_auc", train_roc_auc)
        mlflow.log_metric("test_roc_auc", test_roc_auc)
        
        report = classification_report(y_test, y_pred, output_dict=True)
        for label, metrics in report.items():
            if isinstance(metrics, dict):
                for metric_name, value in metrics.items():
                    mlflow.log_metric(f"{label}_{metric_name}", value)
        
        cm = confusion_matrix(y_test, y_pred)
        mlflow.log_metric("tn", cm[0,0])
        mlflow.log_metric("fp", cm[0,1])
        mlflow.log_metric("fn", cm[1,0])
        mlflow.log_metric("tp", cm[1,1])
        
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
        cv_scores = cross_val_score(pipe, X_train, y_train, cv=skf, scoring='roc_auc')
        mlflow.log_metric("cv_mean_roc_auc", cv_scores.mean())
        mlflow.log_metric("cv_std_roc_auc", cv_scores.std())
        
        mlflow.sklearn.log_model(pipe, artifact_path="adaboost_pipeline")
        
        print(f"ADA n={exp['n_estimators']} lr={exp['learning_rate']} depth={exp['max_depth']} null={exp['null_thresh']} → Train: {train_roc_auc:.4f} Test: {test_roc_auc:.4f} CV: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

2026/05/07 07:05:40 INFO mlflow.tracking.fluent: Experiment with name 'AdaBoost_Training' does not exist. Creating a new experiment.
2026/05/07 07:25:04 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/07 07:25:23 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


ADA n=50 lr=1.0 depth=1 null=0.5 → Train: 0.8463 Test: 0.8452 CV: 0.8447 ± 0.0020
🏃 View run ADA_n=50_lr=1.0_depth=1_null=0.5 at: https://dagshub.com/Nestor-Dzadzamia/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/3/runs/d826991b0040430ea897934116d09bf3
🧪 View experiment at: https://dagshub.com/Nestor-Dzadzamia/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/3


2026/05/07 07:55:52 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/07 07:56:12 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


ADA n=100 lr=1.0 depth=1 null=0.5 → Train: 0.8549 Test: 0.8525 CV: 0.8535 ± 0.0031
🏃 View run ADA_n=100_lr=1.0_depth=1_null=0.5 at: https://dagshub.com/Nestor-Dzadzamia/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/3/runs/691b63498a684638994d3d85fa5b271b
🧪 View experiment at: https://dagshub.com/Nestor-Dzadzamia/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/3


2026/05/07 08:44:47 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/07 08:45:07 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


ADA n=100 lr=0.5 depth=2 null=0.5 → Train: 0.8638 Test: 0.8610 CV: 0.8628 ± 0.0024
🏃 View run ADA_n=100_lr=0.5_depth=2_null=0.5 at: https://dagshub.com/Nestor-Dzadzamia/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/3/runs/8baa16c72e9f4d69b77cf5966fca9bc3
🧪 View experiment at: https://dagshub.com/Nestor-Dzadzamia/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/3
